In [9]:
import pandas as pd
import os
import shutil
import subprocess
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import re
import json

## Include the same epitope data (with all lineage members) but exclude the target antibody data.

In [10]:
# Define file paths
input_file = "/home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_duplicates_geomean.txt"
output_dirs = {
    "50": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_50",
    "1": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_1",
    "0.2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_0.2"
}

# Create output directories if they do not exist
for key in output_dirs:
    os.makedirs(output_dirs[key], exist_ok=True)

In [12]:
# List of antibodies to include
antibodies_to_include = ['561_01_18', 'N6', 'PGT121', 'VRC01',
                         'SF12', '10-1074', 'PGT145', 'M1214_N1', '3BNC117',
                         'PGDM1400', 'VRC38.01', 'PG9', '8ANC195', '35O22', 
                         'VRC34.01', 'PGT135', 'b12', 'VRC-PG04', 'PGT151', 
                         'CH01', 'DH270.6', '4E10', 'PGT128', 'HJ16', 'VRC26.25', 'VRC-CH31', '10E8', 'VRC07-523-W54-LS.v3']

# Print the number of antibodies in the list
print(f"Number of antibodies in the list: {len(antibodies_to_include)}")

Number of antibodies in the list: 28


In [4]:
# Read the input file
df = pd.read_csv(input_file, sep="\t")

# Filter the dataframe to include only specified antibodies
filtered_df = df[df['Antibody'].isin(antibodies_to_include)]

filtered_df

,Antibody,epitope,Virus,clade,IC50,IC80,envseq,heavy,light,lineage
698,10-1074,GP120_V3,0013095_2_11,C,16.54,50.000000,MRVKGILR-NY-QQWW--------IWSIL---GFWMLM--NCNVGG...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
699,10-1074,GP120_V3,001428_2_42,C,0.02,0.168800,MRVRGILR-NY-QQWW--------MWGVL---GFWMLM--ICNGVE...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
700,10-1074,GP120_V3,0041_V3_C18,C,50.00,50.000000,MRVRGILR-NW-QLWW--------TWGIL---GFWMVM--NCNVRG...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
701,10-1074,GP120_V3,0077_V1_C16,C,50.00,50.000000,MRVMGSMR-NC-QRWW--------IWGIL---GFWMLM--TCNMEE...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
702,10-1074,GP120_V3,00836_2_5,C,50.00,50.000000,MRVRGIRR-NY-QHWW--------IWGIL---GFWMLM--ICKGGR...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
...,...,...,...,...,...,...,...,...,...,...
62197,b12,GP120_CD4BS,ZM246F,C,20.09,NaN,MRVMGILR-NC-QQWW--------IWSIL---GF--LM--IYSVIG...,---LEQSGAEVKKPGASVKVSCQ------ASG---YRFS-N-----...,E--LTQAPGTLSLSPGERATFSCRSSHSI------R-S-------R...,unknown
62198,b12,GP120_CD4BS,ZM247_V1,C,50.00,NaN,MRAMGIKR-NC-QRWL--------IWGIL---GFWVLL--TYNVMG...,---LEQSGAEVKKPGASVKVSCQ------ASG---YRFS-N-----...,E--LTQAPGTLSLSPGERATFSCRSSHSI------R-S-------R...,unknown
62199,b12,GP120_CD4BS,ZM249_1,C,3.30,15.933325,MRVMGILR-NC-QPWW--------IWSIL---GFWMLM--BCSG--...,---LEQSGAEVKKPGASVKVSCQ------ASG---YRFS-N-----...,E--LTQAPGTLSLSPGERATFSCRSSHSI------R-S-------R...,unknown
62200,b12,GP120_CD4BS,ZM53_12,C,36.08,50.000000,MRVREIPR-NY-QQWW--------IWGIL---GFWMLM--ICSVVG...,---LEQSGAEVKKPGASVKVSCQ------ASG---YRFS-N-----...,E--LTQAPGTLSLSPGERATFSCRSSHSI------R-S-------R...,unknown


In [6]:
def is_epitope_match(epitope_data, reference_epitope):
    """Returns True if the two epitopes are similar (either identical or one contains the other)."""
    # Handle cases where either value is NaN/None or not a string
    if pd.isna(epitope_data) or pd.isna(reference_epitope):
        return False
    
    # Convert to strings if they aren't already
    epitope_str = str(epitope_data) if not isinstance(epitope_data, str) else epitope_data
    ref_str = str(reference_epitope) if not isinstance(reference_epitope, str) else reference_epitope
    
    return (epitope_str == ref_str or 
            ref_str in epitope_str or 
            epitope_str in ref_str)

# Load the catnap JSON data to get valid Antibody-virus pairs
with open("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold0.2_removed_outliers_duplicates_geomean_with_epitope&lineage.json", 'r') as f:
    catnap_data = json.load(f)

# Create set of valid Antibody-virus pairs from catnap JSON
valid_pairs = set()
for item in catnap_data:
    if len(item) >= 3:  # Ensure item has at least 3 elements
        antibody = item[1]
        virus = item[2]
        valid_pairs.add((antibody, virus))

for antibody, group in filtered_df.groupby("Antibody"):
    # Get the target antibody's epitope
    target_epitope = group['epitope'].iloc[0]
    
    # Skip if target epitope is NaN
    if pd.isna(target_epitope):
        print(f"Antibody: {antibody} has no epitope data, skipping...")
        continue
    
    # Find matching instances
    matched_df = df[
        df['epitope'].apply(lambda x: is_epitope_match(x, target_epitope))
    ]
    
    # Exclude the target antibody itself
    matched_df = matched_df[matched_df['Antibody'] != antibody]
    
    # Further filter to only include pairs that exist in catnap JSON
    matched_df = matched_df[
        matched_df.apply(lambda row: (row['Antibody'], row['Virus']) in valid_pairs, axis=1)
    ]

    # Print the number of included instances
    print(f"Antibody: {antibody}, Epitope: {target_epitope}, Matched instances: {len(matched_df)}")

Antibody: 10-1074, Epitope: GP120_V3 , Matched instances: 10042
Antibody: 10E8, Epitope: GP41_MPER, Matched instances: 6362
Antibody: 35O22, Epitope: GP41-GP41_INTERFACE, Matched instances: 466
Antibody: 3BNC117, Epitope: GP120_CD4BS, Matched instances: 19858
Antibody: 4E10, Epitope: GP41_MPER, Matched instances: 6221
Antibody: 561_01_18, Epitope: GP120_CD4BS, Matched instances: 20553
Antibody: 8ANC195, Epitope: GP41-GP120_INTERFACE, Matched instances: 717
Antibody: CH01, Epitope: GP120_V2 , Matched instances: 11949
Antibody: DH270.6, Epitope: GP120_V3 , Matched instances: 10865
Antibody: HJ16, Epitope: GP120_CD4BS, Matched instances: 20614
Antibody: M1214_N1, Epitope: GP120_CD4BS, Matched instances: 20830
Antibody: N6, Epitope: GP120_CD4BS, Matched instances: 20418
Antibody: PG9, Epitope: GP120_V2 , Matched instances: 11406
Antibody: PGDM1400, Epitope: GP120_V2 , Matched instances: 11167
Antibody: PGT121, Epitope: GP120_V3 , Matched instances: 9673
Antibody: PGT128, Epitope: GP120_V3 

In [7]:
matched_df

,Antibody,epitope,Virus,clade,IC50,IC80,envseq,heavy,light,lineage
3554,12A12,GP120_CD4BS,0013095_2_11,C,0.18,0.970,MRVKGILR-NY-QQWW--------IWSIL---GFWMLM--NCNVGG...,SQHLVQSGTQVKKPGASVRISCQ------ASG---YSFT-D-----...,DIQMTQSPSSLSASVGDRVTITCQAGQGI--------G-------S...,12A12
3555,12A12,GP120_CD4BS,001428_2_42,C,0.02,0.070,MRVRGILR-NY-QQWW--------MWGVL---GFWMLM--ICNGVE...,SQHLVQSGTQVKKPGASVRISCQ------ASG---YSFT-D-----...,DIQMTQSPSSLSASVGDRVTITCQAGQGI--------G-------S...,12A12
3556,12A12,GP120_CD4BS,0077_V1_C16,C,7.00,NaN,MRVMGSMR-NC-QRWW--------IWGIL---GFWMLM--TCNMEE...,SQHLVQSGTQVKKPGASVRISCQ------ASG---YSFT-D-----...,DIQMTQSPSSLSASVGDRVTITCQAGQGI--------G-------S...,12A12
3557,12A12,GP120_CD4BS,00836_2_5,C,50.00,NaN,MRVRGIRR-NY-QHWW--------IWGIL---GFWMLM--ICKGGR...,SQHLVQSGTQVKKPGASVRISCQ------ASG---YSFT-D-----...,DIQMTQSPSSLSASVGDRVTITCQAGQGI--------G-------S...,12A12
3558,12A12,GP120_CD4BS,0260_V5_C36,A1,0.32,NaN,MRVMGIQR-NS-QCFL--------SWGML---VLGIMM--ICSAVG...,SQHLVQSGTQVKKPGASVRISCQ------ASG---YSFT-D-----...,DIQMTQSPSSLSASVGDRVTITCQAGQGI--------G-------S...,12A12
...,...,...,...,...,...,...,...,...,...,...
62606,gVRC-H1dC38/VRC01L,GP120_CD4BS,ZM215_8,C,0.11,0.924,MRVMGILR-NC-QQWW--------IWGIL---GFW--M--ICNVVG...,QVTLVQSGNQLKRPGASVRISCE------TSG---YNFM-D-----...,EIVLTQSPGTLSLSPGETAIISCRTSQYG------S----------...,unknown
62607,gVRC-H1dC38/VRC01L,GP120_CD4BS,ZM233_6,C,50.00,50.000,MRVRGIMR-NW-QQWW--------IWGSL---GFWMLI--ICNVMG...,QVTLVQSGNQLKRPGASVRISCE------TSG---YNFM-D-----...,EIVLTQSPGTLSLSPGETAIISCRTSQYG------S----------...,unknown
62608,gVRC-H1dC38/VRC01L,GP120_CD4BS,ZM249_1,C,0.35,1.540,MRVMGILR-NC-QPWW--------IWSIL---GFWMLM--BCSG--...,QVTLVQSGNQLKRPGASVRISCE------TSG---YNFM-D-----...,EIVLTQSPGTLSLSPGETAIISCRTSQYG------S----------...,unknown
62609,gVRC-H1dC38/VRC01L,GP120_CD4BS,ZM53_12,C,2.53,10.100,MRVREIPR-NY-QQWW--------IWGIL---GFWMLM--ICSVVG...,QVTLVQSGNQLKRPGASVRISCE------TSG---YNFM-D-----...,EIVLTQSPGTLSLSPGETAIISCRTSQYG------S----------...,unknown


In [9]:
unique_df = matched_df[['Antibody', 'epitope', 'Virus']].drop_duplicates().reset_index(drop=True)

unique_df

,Antibody,epitope,Virus
0,12A12,GP120_CD4BS,0013095_2_11
1,12A12,GP120_CD4BS,001428_2_42
2,12A12,GP120_CD4BS,0077_V1_C16
3,12A12,GP120_CD4BS,00836_2_5
4,12A12,GP120_CD4BS,0260_V5_C36
...,...,...,...
20038,gVRC-H1dC38/VRC01L,GP120_CD4BS,ZM215_8
20039,gVRC-H1dC38/VRC01L,GP120_CD4BS,ZM233_6
20040,gVRC-H1dC38/VRC01L,GP120_CD4BS,ZM249_1
20041,gVRC-H1dC38/VRC01L,GP120_CD4BS,ZM53_12


#### For the Python process, we need to modify it to ensure that:
Run MAFFT alignment on row['envseq'] only
Concatenate the post-alignment envseq with row['heavy'] and row['light']
Use this final sequence to generate all downstream files


#### For r_script_path = "/home/yujieq/work/ML_training/bNAb-ReP/scripts/bNAb-ReP_preprocess_h_l_chains.R". We do the following changes:
Heavy/Light Data:
We read a CSV (here assumed to be "heavy_light.csv") that maps each env sequence ID to its heavy and light sequences.

Appending:
For every row in aln.alignment.hxb2.ali (the MAFFT‐aligned env sequences plus HXB2), if the ID is not the HXB2 identifier, we append the corresponding heavy and light sequences. (HXB2 is left unchanged.)

Glycan Conversion & Saving:
The new concatenated sequences (now env + heavy + light) are converted to glycan positions using the original function and then saved using the two output formats (with and without HXB2).

In [10]:
# Function to clean and format sequence
def clean_sequence(seq):
    # Replace any non-letter characters with dashes
    cleaned_seq = re.sub(r'[^A-Za-z]', '-', seq)
    # Replace 'B' with 'N'
    cleaned_seq = cleaned_seq.replace('B', 'N')
    # Replace '*' with '-'
    cleaned_seq = cleaned_seq.replace('*', '-')
    # Replace '#' with 'N'
    cleaned_seq = cleaned_seq.replace('#', 'N')
    return cleaned_seq

def process_and_save_files(df, filtered_df, threshold, threshold_label, mafft_path):
    for antibody, group in filtered_df.groupby("Antibody"):
        # Get the target antibody's epitope
        target_epitope = group['epitope'].iloc[0]
        
        # Skip if target epitope is NaN
        if pd.isna(target_epitope):
            print(f"Antibody: {antibody} has no epitope data, skipping...")
            continue
        
        # Find matching instances
        matched_df = df[
            df['epitope'].apply(lambda x: is_epitope_match(x, target_epitope))
        ]
        
        # Exclude the target antibody itself
        matched_df = matched_df[matched_df['Antibody'] != antibody]
        
        # Further filter to only include pairs that exist in catnap JSON
        matched_df = matched_df[
            matched_df.apply(lambda row: (row['Antibody'], row['Virus']) in valid_pairs, axis=1)
        ]
    
        # Print the number of included instances
        print(f"Antibody: {antibody}, Epitope: {target_epitope}, Matched instances: {len(matched_df)}")

        # Sanitize antibody name
        sanitized_antibody = antibody.replace("/", "_")
        
        # Create directory
        antibody_dir = os.path.join(output_dirs[threshold_label], sanitized_antibody)
        os.makedirs(antibody_dir, exist_ok=True)
        
        sequences = []
        neutralization = []

        # Store heavy/light to do concatenation in R
        # We'll write them to a small CSV: "heavy_light.csv"
        heavy_light_lines = ["id,heavy,light"]

        for _, row in matched_df.iterrows():
            antibody = str(row['Antibody']).replace(" ", "")
            epitope = str(row['epitope']).replace(" ", "")
            virus = row['Virus'].replace(" ", "")
            seq_id = f"{antibody}_{epitope}_{virus}"

            # Clean env sequence
            envseq_cleaned = clean_sequence(row['envseq'])

            # Create SeqRecord
            record = SeqRecord(Seq(envseq_cleaned), id=seq_id, description="")
            sequences.append(record)

            # Neutralization
            ic50 = float(row['IC50'])
            neut_value = 0 if ic50 < threshold else 1
            neutralization.append(str(neut_value))

            # Write heavy/light lines
            # We assume heavy/light are already "aligned" or do not need alignment
            heavy_seq = row['heavy']  # or clean_sequence(row['heavy'])
            light_seq = row['light']  # or clean_sequence(row['light'])
            heavy_light_lines.append(f"{seq_id},{heavy_seq},{light_seq}")

        # Write the final unaligned env FASTA
        # alignment_filename = f"{sanitized_antibody}_IC50_{threshold_label}_env_only.fasta"
        # neutralization_filename = f"{sanitized_antibody}_IC50_{threshold_label}_neutralization.txt"
        # Define the output file paths
        alignment_filename = f"{sanitized_antibody}_IC50_{threshold_label}_alignment.fasta"
        neutralization_filename = f"{sanitized_antibody}_IC50_{threshold_label}_neutralization.txt"
        alignment_file = os.path.join(antibody_dir, alignment_filename)
        neutralization_file = os.path.join(antibody_dir, neutralization_filename)



        with open(alignment_file, "w") as f:
            SeqIO.write(sequences, f, "fasta")

        with open(neutralization_file, "w") as f:
            f.write("\n".join(neutralization))

        # CSV for heavy/light
        heavy_light_path = os.path.join(antibody_dir, "heavy_light.csv")
        with open(heavy_light_path, "w") as f:
            f.write("\n".join(heavy_light_lines))

        # Copy R script into antibody directory
        shutil.copy(r_script_path, antibody_dir)

        # Now call the R script: 
        # Arg1: alignment FASTA (env only)
        # Arg2: neutralization file
        # Arg3: path to MAFFT
        # Arg4: heavy_light CSV
        command = f"Rscript {os.path.basename(r_script_path)} {alignment_filename} {neutralization_filename} {mafft_path} heavy_light.csv"
        # Rscript bNAb-ReP_preprocess_h_l_chains.R SF12_IC50_0.2_env_only.fasta SF12_IC50_0.2_neutralization.txt ~/.conda/envs/deep_learning/bin/mafft heavy_light.csv
        subprocess.run(command, shell=True, cwd=antibody_dir)


In [ ]:
# Run below to generate training file:

process_and_save_files(df, filtered_df, 0.2, "0.2", mafft_path)

# You can also run the all-in-one script here for the same purpose: work/ML_training/bNAb-ReP/scripts/Preprocess_pretrain_with_epitope.py

## Generate the antibody specific training data

In [13]:
# Define file paths
input_file = "/home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_duplicates_geomean.txt"
# output_dirs = {
#     "50": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_50",
#     "1": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_1",
#     "0.2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_0.2"
# }

output_dirs = {
    "50": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining/IC50_50",
    "2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining/IC50_2",
    "1": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining/IC50_1",
    "0.2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining/IC50_0.2"
}

# Create output directories if they do not exist
for key in output_dirs:
    os.makedirs(output_dirs[key], exist_ok=True)

In [14]:
r_script_path = "/home/yujieq/work/ML_training/bNAb-ReP/scripts/bNAb-ReP_preprocess_h_l_chains.R"
# r_script_path = "/home/yujieq/work/software/bNAb-ReP/training/bNAb-ReP_preprocess_v.1.1-1.R"

mafft_path = os.path.expanduser("~/.conda/envs/deep_learning/bin/mafft")

In [15]:
# List of antibodies to include
# antibodies_to_include = ['561_01_18', 'N6', 'PGT121', 'VRC01',
#                          'SF12', '10-1074', 'PGT145', 'M1214_N1', '3BNC117',
#                          'PGDM1400', 'VRC38.01', 'PG9', '8ANC195', '35O22', 
#                          'VRC34.01', 'PGT135', 'b12', 'VRC-PG04', 'PGT151', 
#                          'CH01', 'DH270.6', '4E10', 'PGT128', 'HJ16', 'VRC26.25', 'VRC-CH31', '10E8', 'VRC07-523-W54-LS.v3']
antibodies_to_include = ['561_01_18', 'N6', 'PGT121', 'VRC01',
                         'SF12', '10-1074', 'PGT145', 'M1214_N1', '3BNC117',
                         'PGDM1400']


# Print the number of antibodies in the list
print(f"Number of antibodies in the list: {len(antibodies_to_include)}")

Number of antibodies in the list: 10


In [16]:
# Read the input file
df = pd.read_csv(input_file, sep="\t")

# Filter the dataframe to include only specified antibodies
filtered_df = df[df['Antibody'].isin(antibodies_to_include)]

filtered_df

,Antibody,epitope,Virus,clade,IC50,IC80,envseq,heavy,light,lineage
698,10-1074,GP120_V3,0013095_2_11,C,16.54,50.000000,MRVKGILR-NY-QQWW--------IWSIL---GFWMLM--NCNVGG...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
699,10-1074,GP120_V3,001428_2_42,C,0.02,0.168800,MRVRGILR-NY-QQWW--------MWGVL---GFWMLM--ICNGVE...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
700,10-1074,GP120_V3,0041_V3_C18,C,50.00,50.000000,MRVRGILR-NW-QLWW--------TWGIL---GFWMVM--NCNVRG...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
701,10-1074,GP120_V3,0077_V1_C16,C,50.00,50.000000,MRVMGSMR-NC-QRWW--------IWGIL---GFWMLM--TCNMEE...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
702,10-1074,GP120_V3,00836_2_5,C,50.00,50.000000,MRVRGIRR-NY-QHWW--------IWGIL---GFWMLM--ICKGGR...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
...,...,...,...,...,...,...,...,...,...,...
49656,VRC01,GP120_CD4BS,ZM246F,C,8.91,40.933333,MRVMGILR-NC-QQWW--------IWSIL---GF--LM--IYSVIG...,QVQLVQSGGQMKKPGESMRISCR------ASG---YEFI-D-----...,EIVLTQSPGTLSLSPGETAIISCRTSQYG------S----------...,VRC01
49657,VRC01,GP120_CD4BS,ZM247_V1,C,0.28,1.131667,MRAMGIKR-NC-QRWL--------IWGIL---GFWVLL--TYNVMG...,QVQLVQSGGQMKKPGESMRISCR------ASG---YEFI-D-----...,EIVLTQSPGTLSLSPGETAIISCRTSQYG------S----------...,VRC01
49658,VRC01,GP120_CD4BS,ZM249_1,C,0.08,0.281400,MRVMGILR-NC-QPWW--------IWSIL---GFWMLM--BCSG--...,QVQLVQSGGQMKKPGESMRISCR------ASG---YEFI-D-----...,EIVLTQSPGTLSLSPGETAIISCRTSQYG------S----------...,VRC01
49659,VRC01,GP120_CD4BS,ZM53_12,C,0.90,3.371200,MRVREIPR-NY-QQWW--------IWGIL---GFWMLM--ICSVVG...,QVQLVQSGGQMKKPGESMRISCR------ASG---YEFI-D-----...,EIVLTQSPGTLSLSPGETAIISCRTSQYG------S----------...,VRC01


In [17]:
# First filter out rows with empty/missing IC50 values
filtered_df = filtered_df.dropna(subset=['IC50'])

for antibody, group in filtered_df.groupby("Antibody"):
    print(f"Antibody: {antibody}, Antibody instances: {len(group)}")

Antibody: 10-1074, Antibody instances: 1034
Antibody: 3BNC117, Antibody instances: 1092
Antibody: 561_01_18, Antibody instances: 397
Antibody: M1214_N1, Antibody instances: 120
Antibody: N6, Antibody instances: 532
Antibody: PGDM1400, Antibody instances: 1203
Antibody: PGT121, Antibody instances: 1403
Antibody: PGT145, Antibody instances: 662
Antibody: SF12, Antibody instances: 138
Antibody: VRC01, Antibody instances: 1483


In [18]:
# Function to clean and format sequence
def clean_sequence(seq):
    # Replace any non-letter characters with dashes
    cleaned_seq = re.sub(r'[^A-Za-z]', '-', seq)
    # Replace 'B' with 'N'
    cleaned_seq = cleaned_seq.replace('B', 'N')
    # Replace '*' with '-'
    cleaned_seq = cleaned_seq.replace('*', '-')
    # Replace '#' with 'N'
    cleaned_seq = cleaned_seq.replace('#', 'N')
    return cleaned_seq

def process_and_save_files(filtered_df, threshold, threshold_label, mafft_path):
    for antibody, group in filtered_df.groupby("Antibody"):
        # Process only the data specific to this antibody
        print(f"Processing antibody: {antibody}, Instances: {len(group)}")
        
        # Sanitize antibody name for file naming
        sanitized_antibody = antibody.replace("/", "_")
        
        # Create output directory for this antibody
        antibody_dir = os.path.join(output_dirs[threshold_label], sanitized_antibody)
        os.makedirs(antibody_dir, exist_ok=True)
        
        sequences = []
        neutralization = []
        # Build CSV lines for heavy/light data; header with id, heavy, light
        heavy_light_lines = ["id,heavy,light"]

        # Loop over rows for this antibody (group contains only antibody-specific data)
        for _, row in group.iterrows():
            # Construct a unique sequence ID using antibody, epitope, and virus information.
            antibody_str = str(row['Antibody']).replace(" ", "")
            epitope = str(row['epitope']).replace(" ", "")
            virus = row['Virus'].replace(" ", "")
            seq_id = f"{antibody_str}_{epitope}_{virus}"

            # Clean env sequence
            envseq_cleaned = clean_sequence(row['envseq'])
            
            # Create a SeqRecord for the env sequence
            record = SeqRecord(Seq(envseq_cleaned), id=seq_id, description="")
            sequences.append(record)

            # Determine the neutralization value
            ic50 = float(row['IC50'])
            neut_value = 0 if ic50 < threshold else 1
            neutralization.append(str(neut_value))
            
            # Write heavy and light sequences to CSV line
            heavy_seq = row['heavy']  # assuming already formatted as desired
            light_seq = row['light']
            heavy_light_lines.append(f"{seq_id},{heavy_seq},{light_seq}")

        # Define output file names (alignment FASTA and neutralization text)
        alignment_filename = f"{sanitized_antibody}_IC50_{threshold_label}_antibody_alignment.fasta"
        neutralization_filename = f"{sanitized_antibody}_IC50_{threshold_label}_antibody_neutralization.txt"
        alignment_file = os.path.join(antibody_dir, alignment_filename)
        neutralization_file = os.path.join(antibody_dir, neutralization_filename)

        # Write the env FASTA (to be aligned in R later)
        with open(alignment_file, "w") as f:
            SeqIO.write(sequences, f, "fasta")

        # Write the neutralization values
        with open(neutralization_file, "w") as f:
            f.write("\n".join(neutralization))

        # Write the heavy/light CSV file
        heavy_light_path = os.path.join(antibody_dir, "heavy_light_antibody.csv")
        with open(heavy_light_path, "w") as f:
            f.write("\n".join(heavy_light_lines))

        # Copy the R script to the antibody directory
        shutil.copy(r_script_path, antibody_dir)

        # Call the R script with the alignment FASTA, neutralization file, mafft path, and heavy_light CSV
        command = f"Rscript {os.path.basename(r_script_path)} {alignment_filename} {neutralization_filename} {mafft_path} heavy_light_antibody.csv"
        subprocess.run(command, shell=True, cwd=antibody_dir)


In [19]:
# Process and save files for different thresholds
process_and_save_files(filtered_df, 2, "2", mafft_path)

Processing antibody: 10-1074, Instances: 1034


Read 1034 items
nadd = 1
rescale = 1
dndpre (aa) Version 7.505
alg=X, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

rescale = 1
All-to-all alignment.
 1033 / 1034

##### writing hat3
pairlocalalign (aa) Version 7.505
alg=Y, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

nadd = 1
nthread = 0
blosum 62 / kimura 200
sueff_global = 0.100000
norg = 1034
njobc = 1035
Loading 'hat3' ... 
done.
rescale = 1
Loading 'hat2n' (aligned sequences - new sequences) ... done.
Loading 'hat2i' (aligned sequences) ... done.
cTEP 0 / 1                    

Combining ..
   done.                      

   done.                      

addsingle (aa) Version 7.505
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strategy:
 Multi-INS-full (Not tested.)
 ?

If unsure which option to use, try 'mafft --auto input > output'.
For more information, see 'mafft --help', 'mafft --man' and the mafft page.

The default gap scoring scheme has been changed i

[1] "Perform one-hot encoding using AA21 (20 AA, glycan)"
[1] "Script duration: 0.55 min"
Processing antibody: 3BNC117, Instances: 1092


Read 1092 items
nadd = 1
rescale = 1
dndpre (aa) Version 7.505
alg=X, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

rescale = 1
All-to-all alignment.
 1091 / 1092

##### writing hat3
pairlocalalign (aa) Version 7.505
alg=Y, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

nadd = 1
nthread = 0
blosum 62 / kimura 200
sueff_global = 0.100000
norg = 1092
njobc = 1093
Loading 'hat3' ... 
done.
rescale = 1
Loading 'hat2n' (aligned sequences - new sequences) ... done.
done.ng 'hat2i' (aligned sequences) ... 
cTEP 0 / 1                    

Combining ..
   done.                      

   done.                      

addsingle (aa) Version 7.505
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strategy:
 Multi-INS-full (Not tested.)
 ?

If unsure which option to use, try 'mafft --auto input > output'.
For more information, see 'mafft --help', 'mafft --man' and the mafft page.

The default gap scoring scheme has been changed in ver

[1] "Perform one-hot encoding using AA21 (20 AA, glycan)"
[1] "Script duration: 0.59 min"
Processing antibody: 561_01_18, Instances: 397


...

[1] "Perform one-hot encoding using AA21 (20 AA, glycan)"
[1] "Script duration: 0.37 min"
Processing antibody: SF12, Instances: 138


Read 138 items
nadd = 1
rescale = 1
dndpre (aa) Version 7.505
alg=X, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

rescale = 1
All-to-all alignment.
  137 / 138

##### writing hat3
pairlocalalign (aa) Version 7.505
alg=Y, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

nadd = 1
nthread = 0
blosum 62 / kimura 200
sueff_global = 0.100000
norg = 138
njobc = 139
Loading 'hat3' ... 
done.
rescale = 1
Loading 'hat2n' (aligned sequences - new sequences) ... done.
Loading 'hat2i' (aligned sequences) ... done.
cTEP 0 / 1                    

Combining ..
   done.                      

   done.                      

addsingle (aa) Version 7.505
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


To keep the alignment length, 4 letters were DELETED.
To know the positions of deleted letters, rerun the same command with the --mapout option.

Strategy:
 Multi-INS-full (Not tested.)
 ?

If unsure which option to use, try 'mafft --auto i

[1] "Perform one-hot encoding using AA21 (20 AA, glycan)"
[1] "Script duration: 0.13 min"
Processing antibody: VRC01, Instances: 1483


Read 1483 items
nadd = 1
rescale = 1
dndpre (aa) Version 7.505
alg=X, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

rescale = 1
All-to-all alignment.
 1482 / 1483

##### writing hat3
pairlocalalign (aa) Version 7.505
alg=Y, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

nadd = 1
nthread = 0
blosum 62 / kimura 200
sueff_global = 0.100000
norg = 1483
njobc = 1484
Loading 'hat3' ... 
done.
rescale = 1
Loading 'hat2n' (aligned sequences - new sequences) ... done.
Loading 'hat2i' (aligned sequences) ... done.
cTEP 0 / 1                    

Combining ..
   done.                      

   done.                      

addsingle (aa) Version 7.505
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strategy:
 Multi-INS-full (Not tested.)
 ?

If unsure which option to use, try 'mafft --auto input > output'.
For more information, see 'mafft --help', 'mafft --man' and the mafft page.

The default gap scoring scheme has been changed i

[1] "Perform one-hot encoding using AA21 (20 AA, glycan)"
[1] "Script duration: 0.79 min"


## Results table

In [1]:
import pandas as pd
import os

In [5]:
# output_dirs = {
#     "50": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_50",
#     "1": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_1",
#     "0.2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_adding_pretrain_epitope_only/IC50_0.2"
# }

output_dirs = {
    "50": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining/IC50_50",
    "2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining/IC50_2",
    "1": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining/IC50_1",
    "0.2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining/IC50_0.2"
}

# output_dirs = {
#     "50": "/home/yujieq/work/ML_training/bNAb-ReP/processed_data/IC50_50_updated",
#     "1": "/home/yujieq/work/ML_training/bNAb-ReP/processed_data/IC50_1_updated",
#     "0.2": "/home/yujieq/work/ML_training/bNAb-ReP/processed_data/IC50_0.2_updated",
#     "2": "/home/yujieq/work/ML_training/bNAb-ReP/processed_data/IC50_2"
# }

antibodies_to_include = ['561_01_18', 'N6', 'PGT121', 'VRC01',
                         'SF12', '10-1074', 'PGT145', 'M1214_N1', '3BNC117',
                         'PGDM1400', 'VRC38.01', 'PG9', '8ANC195', '35O22', 
                         'VRC34.01', 'PGT135', 'b12', 'VRC-PG04', 'PGT151', 
                         'CH01', 'DH270.6', '4E10', 'PGT128', 'HJ16', 'VRC26.25', 'VRC-CH31', '10E8', 'VRC07-523-W54-LS.v3']

In [3]:
# Function to check if all required retrain files exist
def check_all_files_exist(antibody_dir, num_files=10):
    for i in range(1, num_files + 1):
        file_path = os.path.join(antibody_dir, f"retrain_{i}.csv")
        if not os.path.exists(file_path):
            return False
    return True

# Function to load and aggregate performance metrics from multiple retrain files
def load_and_aggregate_metrics(antibody_dir, antibody, metrics, num_files=10):
    data = {metric: [] for metric in metrics}
    
    for i in range(1, num_files + 1):
        file_path = os.path.join(antibody_dir, f"retrain_{i}.csv")
        
        df = pd.read_csv(file_path)
        df.rename(columns={'Unnamed: 0': 'Metric'}, inplace=True)

        for metric in metrics:
            if metric in df['Metric'].values:  # Search in Metric column
                metric_row = df[df['Metric'] == metric]
                mean_value = metric_row['mean'].values[0]
                sd_value = metric_row['sd'].values[0]
                data[metric].append((mean_value, sd_value))

    aggregated_data = {}
    for metric in metrics:
        if data[metric]:
            mean_values = [x[0] for x in data[metric]]
            sd_values = [x[1] for x in data[metric]]
            mean_of_means = sum(mean_values) / len(mean_values)
            mean_of_sds = sum(sd_values) / len(sd_values)
            aggregated_data[metric] = (mean_of_means, mean_of_sds)
        else:
            aggregated_data[metric] = (None, None)
    
    return aggregated_data

# Function to process all antibodies and thresholds
def process_all_antibodies(output_dirs, antibodies, metrics):
    results = {threshold: {} for threshold in output_dirs}
    
    for threshold_label, directory in output_dirs.items():
        for antibody in antibodies:
            sanitized_antibody = antibody.replace("/", "_")
            antibody_dir = os.path.join(directory, sanitized_antibody)
            if os.path.exists(antibody_dir) and check_all_files_exist(antibody_dir):
                aggregated_data = load_and_aggregate_metrics(antibody_dir, antibody, metrics)
                if aggregated_data is not None:
                    if threshold_label not in results:
                        results[threshold_label] = {}
                    results[threshold_label][sanitized_antibody] = aggregated_data
    
    return results

In [6]:
# Define the performance metrics to be calculated
performance_metrics = ['accuracy', 'auc', 'mcc']

# Run the processing for all antibodies and thresholds
aggregated_results = process_all_antibodies(output_dirs, antibodies_to_include, performance_metrics)

# Format the results for display
for threshold_label, threshold_data in aggregated_results.items():
    formatted_results = {metric: [] for metric in performance_metrics}
    for antibody, metrics_data in threshold_data.items():
        for metric in performance_metrics:
            if metrics_data[metric]:
                values = metrics_data[metric]
                if values[0] is not None and values[1] is not None:
                    formatted_results[metric].append(f"{values[0]:.2f} ({values[1]:.2f})")
                else:
                    formatted_results[metric].append("N/A")
            else:
                formatted_results[metric].append("N/A")
    

    # Create a DataFrame for the current threshold
    df = pd.DataFrame(formatted_results, index=list(threshold_data.keys()))
    df = df.transpose()
    
    # Save the DataFrame to a CSV file
    csv_filename = f"threshold_{threshold_label}.csv"
    df.to_csv(csv_filename, index=True)
    
    # Display the DataFrame
    print(f"Threshold: {threshold_label}")
    print(df)
    print("\n")

Threshold: 50
            561_01_18           N6       PGT121        VRC01         SF12  \
accuracy  0.81 (0.19)  0.61 (0.29)  0.90 (0.02)  0.93 (0.01)  0.85 (0.08)   
auc       0.76 (0.16)  0.68 (0.22)  0.94 (0.02)  0.89 (0.03)  0.86 (0.08)   
mcc       0.57 (0.19)  0.35 (0.23)  0.78 (0.04)  0.69 (0.05)  0.72 (0.12)   

              10-1074       PGT145     M1214_N1      3BNC117     PGDM1400  \
accuracy  0.96 (0.01)  0.82 (0.04)  0.74 (0.11)  0.93 (0.02)  0.90 (0.02)   
auc       0.97 (0.01)  0.85 (0.04)  0.69 (0.12)  0.91 (0.04)  0.92 (0.02)   
mcc       0.92 (0.02)  0.62 (0.06)  0.57 (0.15)  0.71 (0.07)  0.76 (0.04)   

          ...       PGT151         CH01      DH270.6         4E10  \
accuracy  ...  0.86 (0.04)  0.80 (0.04)  0.92 (0.03)  0.91 (0.04)   
auc       ...  0.88 (0.03)  0.82 (0.05)  0.93 (0.04)  0.78 (0.06)   
mcc       ...  0.71 (0.07)  0.61 (0.08)  0.85 (0.07)  0.50 (0.10)   

               PGT128         HJ16     VRC26.25     VRC-CH31         10E8  \
accuracy  0.88